<h2><center>MNIST classification</center></h2>

In [ ]:
# As a first step, we may want to switch to a GPU-acceperated VM
# In the menu: Runtime->Change runtime type->Hardware Accelerator->GPU.
#
# This will thest if we have a GPU-equipped VM and return some useful system-level information
!nvidia-smi

# Which GNU/Linux distribution is installed on our VM ?
!lsb_release -a

# Which version of the Linux kernel our VM has ?
!uname -a

# How much free memory our VM has ?
!free -h

# Which storage facilities our VM has ?
!mount

# Which python version our VM has installed ?
!python --version

# Importing Torch and related libraries

In [ ]:
# Importing Torch 2.x module
import torch, torchvision
print("Using Torch version " + str(torch.__version__))

In [ ]:
# Torchvision provides datasets and more for CV tasks
import torchvision
print("Using Torch version " + str(torchvision.__version__))

In [ ]:
# Thid is optional for experiment reproducibility
#torch.manual_seed(1)

# Loading and preparing the MNIST dataset

In [ ]:
#These transforms actually perform scaling only
from torchvision.transforms import v2

  # Defining the tranformations we want to apply to our dataset (0-1 normalization in this case)
transform = v2.Compose([
      v2.ToImage(),
      v2.ToDtype(torch.float32, scale=True)
  ])

  # Training-only transformations: data augmentation (random horizontal flip and small shifts)
train_transform = v2.Compose([
      v2.ToImage(),
      v2.RandomHorizontalFlip(),
      v2.RandomAffine(degrees=0, translate=(0.1, 0.1)),
      v2.ToDtype(torch.float32, scale=True)
  ])


In [ ]:
# Loading into memory the datasets for training and testing applying scaling
from torchvision import datasets

training_set = datasets.FashionMNIST('./data', train=True, transform=train_transform, download=True)
test_set = datasets.FashionMNIST('./data', train=False, transform=transform, download=True)


Creating a dataloder for the train and test datasets

In [ ]:
from torch.utils.data import DataLoader

# Batch sizes are usually multiples of 2 in the 16-32 range
batch_size = 32

# Creating one data loader for the training set and one for the test set
train_dataloader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_set, batch_size=batch_size,shuffle=False)

# Inspecting the train images and labels...
for X, y in train_dataloader:
    print(f"Shape of train images [N, C, H, W]: {X.shape}")
    print(f"Shape of train labels: {y.shape} {y.dtype}")
    break

# ... and the test images and labels
for X, y in test_dataloader:
    print(f"Shape of train images [N, C, H, W]: {X.shape}")
    print(f"Shape of train labels: {y.shape} {y.dtype}")
    break

In [ ]:
# Let us visualize the first training sample in the batch using the Gnuplot library
from matplotlib import pyplot as plt

for X, y in train_dataloader:
    plt.imshow(X[0].squeeze(), cmap='gray')
    plt.title(f"Label: {y[0].item()}")
    plt.show()
    break


# Defining the Model

In [ ]:
from torch import nn

# Define a LeNet5-like model
# c1, c2 = number of filters in the first and second convolutional layer
class MyModel(nn.Module):
    def __init__(self, c1=16, c2=32):
        super().__init__()
        # Convolutional layers (feature extraction)
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=c1, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=c1, out_channels=c2, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        # Fully connected layers (classification)
        self.classifier = nn.Sequential(
            nn.Linear(c2*7*7, 100),
            nn.ReLU(),
            nn.Dropout(0.5),  # randomly drops 50% of the neurons during training
            nn.Linear(100, 10),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        logits = self.classifier(x)
        return logits

# This is the model instance that will be trained and tested
model = MyModel(c1=16, c2=32)
# Number of learnable parameters
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Getting hold of our GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# The model is bound to the GPU
model = model.to(device)
print(model)

# Instantiate one loss function and an optimizer

In [ ]:

#Since we have a LogSoftMax layer in our model, we use NLLLoss instead of CrossEntropyLoss
loss_fn = nn.NLLLoss()
# The optimizers module provides a number of optimization algorithms for updating
# a netwok parameters accoridng to the computed error gradints
# momentum smooths the updates, weight_decay is the L2 regularization term
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=5e-4)
# The learning rate is divided by 10 every 10 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Defnining the training function

In [ ]:
# This is where the actual training-testing happens
def train_epoch(dataloader, model, loss_fn, optimizer, verbose = False):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred,y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if verbose and batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

# Defining the testing function

In [ ]:
def eval_epoch(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X,y = X.to(device), y.to(device)
            pred=model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return test_loss, correct*100
    #print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

#Main training loop - here is where learning happens

In [ ]:
#For being able to plot the training history afterwards
history = {}
history['train_loss'] = []
history['train_acc'] = []
history['test_loss'] = []
history['test_acc'] = []

# Best test accuracy reached so far
best_acc = 0

epochs = 20
for e in range(epochs):
    # Updating the model parameters for one epoch
    train_epoch(train_dataloader, model, loss_fn, optimizer)
    # Updating the learning rate according to the scheduler
    scheduler.step()
    # Testing on the train and testing sets to evaluate the model generalization ability
    train_loss, train_acc = eval_epoch(train_dataloader, model, loss_fn)
    test_loss, test_acc = eval_epoch(test_dataloader, model, loss_fn)

    # Saving the model with the best test accuracy
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_model.pt')

    print ('Epoch %d / %d lossTrain %.3f lossTest %.3f accuracyTrain %.3f accuracyTest %.3f' %(int(e)+1, epochs, train_loss, test_loss, train_acc, test_acc))
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

print("Done! Best test accuracy: %.2f%%" % best_acc)

Epoch 1 / 20 lossTrain 0.570 lossTest 0.533 accuracyTrain 77.083 accuracyTest 78.340
Epoch 2 / 20 lossTrain 0.456 lossTest 0.424 accuracyTrain 82.458 accuracyTest 84.240
Epoch 3 / 20 lossTrain 0.408 lossTest 0.390 accuracyTrain 84.677 accuracyTest 85.750
Epoch 4 / 20 lossTrain 0.374 lossTest 0.361 accuracyTrain 85.978 accuracyTest 86.640
Epoch 5 / 20 lossTrain 0.360 lossTest 0.343 accuracyTrain 86.315 accuracyTest 87.410
Epoch 6 / 20 lossTrain 0.341 lossTest 0.331 accuracyTrain 87.093 accuracyTest 88.110
Epoch 7 / 20 lossTrain 0.320 lossTest 0.308 accuracyTrain 87.882 accuracyTest 88.740
Epoch 8 / 20 lossTrain 0.345 lossTest 0.341 accuracyTrain 86.660 accuracyTest 86.940
Epoch 9 / 20 lossTrain 0.318 lossTest 0.310 accuracyTrain 87.937 accuracyTest 88.390
Epoch 10 / 20 lossTrain 0.310 lossTest 0.300 accuracyTrain 88.235 accuracyTest 89.040
Epoch 11 / 20 lossTrain 0.277 lossTest 0.274 accuracyTrain 89.657 accuracyTest 90.150
Epoch 12 / 20 lossTrain 0.271 lossTest 0.271 accuracyTrain 89.8

**Note:** this run (16/32 filters, data augmentation, dropout, L2, momentum, StepLR) was stopped after epoch 17 of 20. Best test accuracy: **90.75%** (epoch 17), with train and test accuracy within ~0.5% of each other (no overfitting).

# Visualizing the network performance

In [ ]:
# We plot now the train and test loss and accuracy curves from history.history.keys()
_, axes = plt.subplots(nrows=2, ncols=1)

# Plotting the loss graph
axes[0].set_title("Training report")
axes[0].grid()
axes[0].plot(history['train_loss'])
axes[0].plot(history['test_loss'])
axes[0].set_ylabel=('Loss [bit]')
axes[0].legend(['Train', 'Test'], loc='upper right')

# Plotting the accuracy graph
axes[1].grid()
axes[1].plot(history['train_acc'])
axes[1].plot(history['test_acc'])
axes[1].set_ylabel('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend(['Train', 'Test'], loc='lower right')